# 01 — Data Preparation and Text Chunking

## Project Title

Analisis Topik Pidato Presiden Prabowo pada Sidang Umum Perserikatan Bangsa-Bangsa Menggunakan Manual Analysis dan BERTopic

## Objective

Notebook ini bertujuan untuk menyiapkan data teks pidato sebelum dianalisis menggunakan Manual Analysis dan BERTopic. Karena data hanya terdiri dari satu pidato utuh, teks perlu dipecah menjadi beberapa unit analisis kecil atau text chunks.

Setiap chunk terdiri dari 3 kalimat agar konteks semantik tetap terjaga dan dapat digunakan sebagai pseudo-document dalam proses topic modeling.

## Mengapa Text Chunking Diperlukan?

BERTopic umumnya bekerja pada kumpulan dokumen. Pada project ini, data yang digunakan hanya berupa satu pidato utuh. Jika pidato dimasukkan sebagai satu dokumen, model tidak memiliki cukup variasi unit analisis untuk menemukan pola topik.

Oleh karena itu, pidato dipecah menjadi beberapa chunk. Setiap chunk diperlakukan sebagai dokumen kecil yang tetap merepresentasikan bagian tertentu dari pidato.

Strategi chunking setiap 3 kalimat dipilih karena:

1. Satu kalimat sering kali terlalu pendek untuk membawa konteks topik.
2. Tiga kalimat cukup untuk menangkap satu gagasan utama.
3. Chunk yang terlalu panjang dapat mencampurkan beberapa topik sekaligus.
4. Struktur analisis tetap dapat ditelusuri melalui nomor kalimat awal dan akhir.

In [1]:
# ============================================================
# IMPORT LIBRARY DASAR
# ============================================================

import re
from pathlib import Path

import pandas as pd
import numpy as np

from IPython.display import display

pd.set_option("display.max_colwidth", 200)
pd.set_option("display.max_rows", 100)
pd.set_option("display.width", 120)

In [2]:
# ============================================================
# KONFIGURASI PATH PROJECT
# ============================================================

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
FINAL_DIR = PROJECT_ROOT / "data" / "final"

RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
FINAL_DIR.mkdir(parents=True, exist_ok=True)

print("Project root :", PROJECT_ROOT)
print("Raw dir      :", RAW_DIR)
print("Processed dir:", PROCESSED_DIR)
print("Final dir    :", FINAL_DIR)

Project root : D:\DATA-KAMIL\MATKUL\SEMESTER-1\DATA-MINING\topic-modeling-prabowo-un-speech
Raw dir      : D:\DATA-KAMIL\MATKUL\SEMESTER-1\DATA-MINING\topic-modeling-prabowo-un-speech\data\raw
Processed dir: D:\DATA-KAMIL\MATKUL\SEMESTER-1\DATA-MINING\topic-modeling-prabowo-un-speech\data\processed
Final dir    : D:\DATA-KAMIL\MATKUL\SEMESTER-1\DATA-MINING\topic-modeling-prabowo-un-speech\data\final


In [4]:
# ============================================================
# LOAD TEKS PIDATO
# ============================================================

speech_file = RAW_DIR / "NASKAH_PIDATO_PRABOWO_PBB_80.txt"

if not speech_file.exists():
    raise FileNotFoundError(
        f"File tidak ditemukan: {speech_file}\n"
        "Pastikan file naskah pidato sudah diletakkan di folder data/raw."
    )

raw_text = speech_file.read_text(encoding="utf-8")

print("File berhasil dibaca.")
print("Nama file:", speech_file.name)
print("Jumlah karakter awal:", len(raw_text))

File berhasil dibaca.
Nama file: NASKAH_PIDATO_PRABOWO_PBB_80.txt
Jumlah karakter awal: 11116


In [5]:
# ============================================================
# PREVIEW TEKS
# ============================================================

print("Preview 1.500 karakter pertama:\n")
print(raw_text[:1500])

Preview 1.500 karakter pertama:

His Excellency Mr. Antoni Guterres, Secretary General of the United Nations;
Her Excellency Madam Annalena Baerbock, President of the United Nations General Assembly;
His Excellency Mr. Movses Abelian, Under-Secretary-General for General Assembly and Management;
Excellencies, Head of States, Head of Governments, Distinguished Delegates;
Ladies and Gentlemen.

It is indeed a great honor to stand in this august General Assembly Hall, among leaders and representatives who represent almost all of humanity. We differ in race, religion, and nationality, yet we gather together today as one human family. We are here first and foremost as fellow human beings — each created equal, endowed with unalienable rights to life, liberty, and the pursuit of happiness.

The words of the United Nations Declaration of Independence have inspired democratic movements across continents — including the French Revolution, the Russian Revolution, the Mexican Revolution, the Chines

In [6]:
# ============================================================
# FUNGSI CLEANING RINGAN
# ============================================================

def clean_text_light(text):
    """
    Membersihkan teks secara ringan tanpa menghilangkan konteks semantik.

    Cleaning tidak dibuat agresif karena teks akan digunakan untuk BERTopic.
    Oleh karena itu, tidak dilakukan stopword removal, stemming, atau lowercasing penuh.
    """
    
    text = text.replace("\ufeff", "")
    text = text.replace("\u00a0", " ")
    
    text = text.replace("–", "-")
    text = text.replace("—", "-")
    
    text = re.sub(r"\[([^\]]+)\]", r"\1", text)
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    
    return text.strip()


clean_text = clean_text_light(raw_text)

print("Jumlah karakter setelah cleaning:", len(clean_text))

Jumlah karakter setelah cleaning: 11114


In [7]:
# ============================================================
# FUNGSI PENGHITUNGAN KATA DAN KALIMAT
# ============================================================

def count_words(text):
    """
    Menghitung jumlah kata dengan tetap mempertahankan kata ber-apostrophe
    dan kata ber-hyphen seperti world's dan self-sufficient.
    """
    
    word_pattern = r"\b[A-Za-zÀ-ÖØ-öø-ÿ0-9]+(?:[-'’][A-Za-zÀ-ÖØ-öø-ÿ0-9]+)*\b"
    return len(re.findall(word_pattern, text))


def split_sentences(text):
    """
    Memecah teks menjadi kalimat menggunakan regex sederhana.
    Beberapa abbreviation umum dilindungi agar tidak salah dipotong.
    """
    
    abbreviations = [
        "Mr.", "Mrs.", "Ms.", "Dr.", "Prof.", "Sr.", "Jr.", "St.", "No.",
        "e.g.", "i.e.", "U.S.", "U.K.", "U.N."
    ]
    
    placeholder = "<DOT>"
    protected_text = text
    
    for abbr in abbreviations:
        protected_text = protected_text.replace(abbr, abbr.replace(".", placeholder))
    
    protected_text = re.sub(r"\s+", " ", protected_text).strip()
    
    sentence_candidates = re.split(
        r'(?<=[.!?])\s+(?=[A-Z0-9"“])',
        protected_text
    )
    
    sentences = [
        sentence.replace(placeholder, ".").strip()
        for sentence in sentence_candidates
        if sentence.strip()
    ]
    
    return sentences

In [8]:
# ============================================================
# STATISTIK AWAL TEKS
# ============================================================

sentences = split_sentences(clean_text)

text_stats = {
    "source_file": speech_file.name,
    "raw_char_count": len(raw_text),
    "clean_char_count": len(clean_text),
    "word_count": count_words(clean_text),
    "sentence_count": len(sentences)
}

df_text_stats = pd.DataFrame([text_stats])
display(df_text_stats)

,source_file,raw_char_count,clean_char_count,word_count,sentence_count
0,NASKAH_PIDATO_PRABOWO_PBB_80.txt,11116,11114,1824,118


In [9]:
# ============================================================
# MEMBUAT DATAFRAME KALIMAT
# ============================================================

df_sentences = pd.DataFrame({
    "sentence_id": range(1, len(sentences) + 1),
    "sentence_text": sentences,
    "char_count": [len(sentence) for sentence in sentences],
    "word_count": [count_words(sentence) for sentence in sentences]
})

display(df_sentences.head(10))
display(df_sentences.tail(10))

,sentence_id,sentence_text,char_count,word_count
0,1,"His Excellency Mr. Antoni Guterres, Secretary General of the United Nations; Her Excellency Madam Annalena Baerbock, President of the United Nations General Assembly; His Excellency Mr. Movses Abe...",360,46
1,2,"It is indeed a great honor to stand in this august General Assembly Hall, among leaders and representatives who represent almost all of humanity.",145,24
2,3,"We differ in race, religion, and nationality, yet we gather together today as one human family.",95,16
3,4,"We are here first and foremost as fellow human beings - each created equal, endowed with unalienable rights to life, liberty, and the pursuit of happiness.",155,25
4,5,"The words of the United Nations Declaration of Independence have inspired democratic movements across continents - including the French Revolution, the Russian Revolution, the Mexican Revolution, ...",272,36
5,6,It also gave birth to the Universal Declaration of Human Rights adopted by United Nations in 1948.,98,17
6,7,“All men are created equal” was the creed that opened the way to unprecedented global prosperity and dignity.,109,18
7,8,"And yet, in our own era of scientific and technological triumphs - an era capable of ending hunger, poverty, and environmental ruin - we also continue to face today grave dangerous challenges and ...",210,32
8,9,"Human folly, fueled by fear, racism, hatred, oppression, and apartheid, threatens our common future.",100,14
9,10,My country knows this pain.,27,5


,sentence_id,sentence_text,char_count,word_count
108,109,Is this a dream?,16,4
109,110,"Maybe, but this is the beautiful dream that we must work together towards.",74,13
110,111,Let us work towards this noble goal.,36,7
111,112,Let us continue humanity’s journey of hope - a journey started by our forefathers.,82,13
112,113,A journey that we must complete.,32,6
113,114,Thank you.,10,2
114,115,"Wassalamu’alaikum warahmatullahi wabarakatuh, Syalom, Om santi santi santi om, Namo Buddhaya, Thank you very much.",114,15
115,116,May God bless us all.,21,5
116,117,May peace be upon us.,21,5
117,118,Thank you very much.,20,4


In [10]:
# ============================================================
# FUNGSI TEXT CHUNKING
# ============================================================

def create_chunks(sentences, chunk_size=3, min_last_chunk_words=20):
    """
    Membuat chunk berdasarkan jumlah kalimat tertentu.
    
    Jika chunk terakhir terlalu pendek, chunk tersebut digabung dengan chunk sebelumnya
    agar konteks semantiknya tidak terlalu lemah.
    """
    
    records = []
    
    for start_idx in range(0, len(sentences), chunk_size):
        end_idx = min(start_idx + chunk_size, len(sentences))
        selected_sentences = sentences[start_idx:end_idx]
        chunk_text = " ".join(selected_sentences)
        
        records.append({
            "chunk_id": len(records) + 1,
            "sentence_start": start_idx + 1,
            "sentence_end": end_idx,
            "sentence_count": len(selected_sentences),
            "chunk_text": chunk_text,
            "char_count": len(chunk_text),
            "word_count": count_words(chunk_text)
        })
    
    df_chunks = pd.DataFrame(records)
    
    if len(df_chunks) > 1:
        last_word_count = df_chunks.iloc[-1]["word_count"]
        
        if last_word_count < min_last_chunk_words:
            prev_idx = df_chunks.index[-2]
            last_idx = df_chunks.index[-1]
            
            merged_text = df_chunks.at[prev_idx, "chunk_text"] + " " + df_chunks.at[last_idx, "chunk_text"]
            
            df_chunks.at[prev_idx, "sentence_end"] = df_chunks.at[last_idx, "sentence_end"]
            df_chunks.at[prev_idx, "sentence_count"] += df_chunks.at[last_idx, "sentence_count"]
            df_chunks.at[prev_idx, "chunk_text"] = merged_text
            df_chunks.at[prev_idx, "char_count"] = len(merged_text)
            df_chunks.at[prev_idx, "word_count"] = count_words(merged_text)
            
            df_chunks = df_chunks.iloc[:-1].copy()
            df_chunks["chunk_id"] = range(1, len(df_chunks) + 1)
    
    return df_chunks

In [11]:
# ============================================================
# TEXT CHUNKING SETIAP 3 KALIMAT
# ============================================================

CHUNK_SIZE = 3

df_chunks = create_chunks(
    sentences=sentences,
    chunk_size=CHUNK_SIZE,
    min_last_chunk_words=20
)

print("Jumlah kalimat:", len(sentences))
print("Jumlah chunk  :", len(df_chunks))

display(df_chunks.head(10))

Jumlah kalimat: 118
Jumlah chunk  : 39


,chunk_id,sentence_start,sentence_end,sentence_count,chunk_text,char_count,word_count
0,1,1,3,3,"His Excellency Mr. Antoni Guterres, Secretary General of the United Nations; Her Excellency Madam Annalena Baerbock, President of the United Nations General Assembly; His Excellency Mr. Movses Abe...",602,86
1,2,4,6,3,"We are here first and foremost as fellow human beings - each created equal, endowed with unalienable rights to life, liberty, and the pursuit of happiness. The words of the United Nations Declarat...",527,78
2,3,7,9,3,"“All men are created equal” was the creed that opened the way to unprecedented global prosperity and dignity. And yet, in our own era of scientific and technological triumphs - an era capable of e...",421,64
3,4,10,12,3,"My country knows this pain. For centuries, Indonesians lived under colonial domination, oppression, and slavery. We were treated less than dogs in our own homeland.",164,25
4,5,13,15,3,"We Indonesians know what it means to be denied justice and what it means to live in apartheid, to live in poverty, and to be denied equal opportunity. We also knew what solidarity can do. In our s...",346,60
5,6,16,18,3,"Decisions made here based on human solidarity - by the Security Council and this Assembly - gave Indonesia independence international legitimacy, opened doors, and supported our early development ...",581,83
6,7,19,21,3,"Everyday we witness suffering, genocide, and a blatant disregard for international law and human decency. In the face of these challenges, we must not give up. As the United Nations Secretary Gene...",224,37
7,8,22,24,3,"We cannot surrender our hopes or our ideals. We must draw closer, not drift further apart. Together we must strive to achieve our hopes, our dreams.",148,26
8,9,25,27,3,"The United Nations was born from the ashes of the Second World War that claimed scores of millions of lives. It was created to secure peace, security, justice, and freedom for all. We remain commi...",302,48
9,10,28,30,3,"Today, Indonesia is nearer than ever before to meeting the Sustainable Development Goals of ending extreme poverty and hunger - because years ago this very chamber chose to listen and uphold socia...",358,58


In [12]:
# ============================================================
# VALIDASI KUALITAS CHUNK
# ============================================================

display(df_chunks["word_count"].describe().to_frame(name="word_count_summary"))

short_chunks = df_chunks[df_chunks["word_count"] < 20]

print("Jumlah chunk dengan word_count < 20:", len(short_chunks))

if len(short_chunks) > 0:
    display(short_chunks[["chunk_id", "sentence_start", "sentence_end", "word_count", "chunk_text"]])
else:
    print("Tidak ada chunk yang terlalu pendek.")

,word_count_summary
count,39.000000
mean,46.769231
std,21.020618
min,16.000000
25%,28.000000
50%,48.000000
75%,61.000000
max,88.000000


Jumlah chunk dengan word_count < 20: 2


,chunk_id,sentence_start,sentence_end,word_count,chunk_text
26,27,79,81,16,Who will save them? Who will save the innocent. Who will save the old and women?
28,29,85,87,17,They are dying of starvation. Can we remain silent? Will there be no answer to their screams?
